# MCP Agent — Managed vs Clone

Same agent, same code. Only the **URL** changes.

```
Managed:  https://workspace.cloud.databricks.com/api/2.0/mcp/genie/{space_id}
Clone:    https://app.aws.databricksapps.com/api/2.0/mcp/genie/{gateway_id}
```

In [0]:
%pip install 'openai-agents[mcp]' -q

In [0]:
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
WORKSPACE = w.config.host.rstrip("/")
TOKEN     = dbutils.secrets.get(scope="genie-cache", key="oauth_token")
APP       = dbutils.widgets.get("app_url")
SPACE_ID  = dbutils.widgets.get("space_id")
GATEWAY   = dbutils.widgets.get("gateway_id")
MODEL     = dbutils.widgets.get("model")

MANAGED_URL = f"{WORKSPACE}/api/2.0/mcp/genie/{SPACE_ID}"
CLONE_URL   = f"{APP}/api/2.0/mcp/genie/{GATEWAY}"

print(f"Managed: {MANAGED_URL}")
print(f"Clone:   {CLONE_URL}")

In [0]:
import asyncio, time, logging
from openai import AsyncOpenAI
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStreamableHttp

logging.getLogger("openai.agents").setLevel(logging.ERROR)

# LLM via Databricks FMAPI
oai = AsyncOpenAI(base_url=f"{WORKSPACE}/serving-endpoints", api_key=TOKEN)
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=oai)

AUTH = {"Authorization": f"Bearer {TOKEN}"}

async def ask(url, question):
    async with MCPServerStreamableHttp(params={"url": url, "headers": AUTH}, client_session_timeout_seconds=120) as mcp:
        agent = Agent(name="analyst", model=model, mcp_servers=[mcp])
        t0 = time.time()
        result = await Runner.run(agent, question, max_turns=30) 
        elapsed = round(time.time() - t0, 1)
        print(f"  [{elapsed}s] {result.final_output[:200]}")
        return result, elapsed

print("Ready.")

## A — Managed Databricks MCP

In [0]:
Q = "What are the bottom 3 nations by total revenue?"

print(f"URL: {MANAGED_URL}")
result_a, time_a = await ask(MANAGED_URL, Q)

## B — Clone MCP (1st call)

In [0]:
print(f"URL: {CLONE_URL}")
result_b, time_b = await ask(CLONE_URL, Q)

## C — Clone MCP (2nd call — cache hit)

In [0]:
print(f"URL: {CLONE_URL}")
result_c, time_c = await ask(CLONE_URL, Q)

In [0]:
print(f"A — Managed:     {time_a}s")
print(f"B — Clone 1st:   {time_b}s")
print(f"C — Clone 2nd:   {time_c}s  (cache hit)")
print()
print(f"Same agent. Only the URL changed.")